# Metaheuristics (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/metaheuristics_python.ipynb)

Executable notebook of the chapter “Metaheuristics” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the metaheuristics of the chapter: simulated annealing, tabu search, iterated local search, variable neighbourhood search, GRASP, large neighbourhood search and the genetic and memetic algorithm, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/metaheuristics/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/metaheuristics_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. Nothing to install: only the Python standard library is used.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Random numbers

The random numbers are produced by `Rand`, the Park–Miller minimal standard generator. The structure `Random` contains the seed and, optionally, a list of given numbers, which are returned instead: the exercises give their random numbers explicitly.

In [1]:
# Metaheuristics for the KP, the BPP, the TSP and P||Cmax.
# Items, jobs and cities are indexed 0, 1, ..., n-1; machines 0, ..., m-1; bins 0, ..., nb-1.
# A TSP tour is a permutation pi[0..n-1] of the cities with pi[0] = 0 (the first city).
# Solutions given as input and improved (pi, mach, x, bin) are modified in place;
# solutions only built by a function are returned.

import math


class Random:
    """Random numbers: Park-Miller minimal standard generator, or the numbers of the
    list "given" (returned in order); used counts the numbers drawn so far."""

    def __init__(self, seed=1, given=None):
        self.seed = seed
        self.given = given if given is not None else []
        self.used = 0


def rand(r):
    """Park-Miller minimal standard generator: seed = 16807 * seed mod (2^31 - 1),
    u = seed / (2^31 - 1); if a list of numbers is given, they are returned in order instead."""
    if len(r.given) > 0:
        u = r.given[r.used]
    else:
        r.seed = (16807 * r.seed) % 2147483647
        u = r.seed / 2147483647.0
    r.used = r.used + 1
    return u

## Local searches and auxiliary functions

The 2-opt local search for the TSP, with the $O(1)$ evaluation of a move; the move descent for $P\,||\,C_{\max}$ with the lexicographic comparison, and the LPT starting solution; the local search for the KP with feasible flips and swaps.

In [2]:
def tour_length(pi, d):
    """Length L(pi) of the tour pi."""
    n = len(pi)
    length = 0
    for k in range(n):
        length = length + d[pi[k]][pi[(k + 1) % n]]
    return length


def two_opt_delta(pi, d, i, j):
    """Change of length of the 2-opt move (i,j): the edges (pi[i],pi[i+1]) and (pi[j],pi[j+1])
    are replaced by (pi[i],pi[j]) and (pi[i+1],pi[j+1]); computed in O(1)."""
    n = len(pi)
    a, b, c, e = pi[i], pi[i + 1], pi[j], pi[(j + 1) % n]
    return d[a][c] + d[b][e] - d[a][b] - d[c][e]


def apply_two_opt(pi, i, j):
    """Performs the 2-opt move (i,j): reverses the segment pi[i+1..j] (pi modified in place)."""
    lo, hi = i + 1, j
    while lo < hi:
        pi[lo], pi[hi] = pi[hi], pi[lo]
        lo = lo + 1
        hi = hi - 1

In [3]:
def two_opt_ls(pi, d):
    """2-opt local search with best improvement: stops at a local optimum (pi modified in place).
    Ties between moves are broken in favour of the first move (i,j) in lexicographic order."""
    n = len(pi)
    while True:
        bdelta, bi, bj = 0, -1, -1
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:
                    continue                     # same two edges: no change
                delta = two_opt_delta(pi, d, i, j)
                if delta < bdelta:
                    bdelta, bi, bj = delta, i, j
        # no improving move: local optimum
        if bi < 0:
            return
        apply_two_opt(pi, bi, bj)


def nearest_neighbour(d):
    """Nearest neighbour tour from city 0 (ties: smallest index); returns the tour."""
    n = len(d)
    pi = [0]
    visited = [False] * n
    visited[0] = True
    for k in range(1, n):
        last, nxt = pi[-1], -1
        for j in range(n):
            if not visited[j] and (nxt < 0 or d[last][j] < d[last][nxt]):
                nxt = j
        pi.append(nxt)
        visited[nxt] = True
    return pi

In [4]:
def loads(mach, t, m):
    """Loads L[h] of the machines for the assignment mach."""
    L = [0] * m
    for j in range(len(t)):
        L[mach[j]] = L[mach[j]] + t[j]
    return L


def lex_less(L1, L2):
    """True if the loads L1 are better than L2 in the lexicographic order
    (makespan first, then sum of the squared loads)."""
    C1, C2, S1, S2 = 0, 0, 0, 0
    for h in range(len(L1)):
        if L1[h] > C1:
            C1 = L1[h]
        if L2[h] > C2:
            C2 = L2[h]
        S1 = S1 + L1[h] * L1[h]
        S2 = S2 + L2[h] * L2[h]
    return C1 < C2 or (C1 == C2 and S1 < S2)


def move_descent(t, m, mach):
    """Local search for P||Cmax with the move neighbourhood and best improvement, with respect
    to the lexicographic order (makespan, sum of squared loads); mach modified in place."""
    n = len(t)
    L = loads(mach, t, m)
    while True:
        bj, bh = -1, -1
        bL = L
        for j in range(n):
            for h in range(m):
                if h == mach[j]:
                    continue
                L2 = L[:]
                L2[mach[j]] = L2[mach[j]] - t[j]
                L2[h] = L2[h] + t[j]
                if lex_less(L2, bL):
                    bj, bh, bL = j, h, L2
        if bj < 0:
            return                               # local optimum
        mach[bj] = bh
        L = bL


def lpt(t, m):
    """Longest processing time first: jobs by non-increasing t (ties: smallest index), each
    assigned to the least loaded machine (ties: smallest index); returns the assignment mach."""
    n = len(t)
    order = [0] * n
    mach = [0] * n
    L = [0] * m
    for j in range(n):
        order[j] = j
    for a in range(1, n):                        # insertion sort, stable
        j, b = order[a], a
        while b > 0 and t[order[b - 1]] < t[j]:
            order[b] = order[b - 1]
            b = b - 1
        order[b] = j
    for a in range(n):
        j, h = order[a], 0
        for k in range(1, m):
            if L[k] < L[h]:
                h = k
        mach[j] = h
        L[h] = L[h] + t[j]
    return mach

In [5]:
def profit(p, x):
    """Total profit of the items selected by x."""
    P = 0
    for i in range(len(p)):
        if x[i] == 1:
            P = P + p[i]
    return P


def weight(w, x):
    """Total weight of the items selected by x."""
    W = 0
    for i in range(len(w)):
        if x[i] == 1:
            W = W + w[i]
    return W


def kp_local_search(p, w, c, x):
    """Local search for the KP with best improvement on the feasible neighbours of the flip
    neighbourhood (only additions can improve) and of the swap neighbourhood; x modified in place."""
    n = len(p)
    while True:
        W, P = weight(w, x), profit(p, x)
        # bi = -1: add item bk; else swap bi out, bk in
        bP, bi, bk = P, -1, -1
        for k in range(n):
            if x[k] == 0 and W + w[k] <= c and P + p[k] > bP:
                bP, bi, bk = P + p[k], -1, k
        for i in range(n):
            if x[i] != 1:
                continue
            for k in range(n):
                if x[k] == 0 and W - w[i] + w[k] <= c and P - p[i] + p[k] > bP:
                    bP, bi, bk = P - p[i] + p[k], i, k
        if bk < 0:
            return                               # local optimum
        if bi >= 0:
            x[bi] = 0
        x[bk] = 1

## Simulated annealing

Simulated annealing for the TSP, with the 2-opt neighbourhood, and for the KP with the penalized objective.

In [6]:
def simulated_annealing_tsp(d, pi, T0, alpha, L, Tmin, r, curve=None):
    """Simulated annealing for the TSP with the 2-opt neighbourhood and geometric cooling.
    pi: starting tour on input, best tour found on output (modified in place); returns its length.
    If curve is not None, the length of the current tour after each move is appended to it."""
    n = len(pi)
    # the M = n(n-3)/2 moves (i,j) of 2-opt
    mi, mj = [], []
    for i in range(n - 1):
        for j in range(i + 2, n):
            if not (i == 0 and j == n - 1):
                mi.append(i)
                mj.append(j)
    M = len(mi)

    cur = pi[:]                                  # current solution
    fcur = tour_length(cur, d)
    best = cur[:]                                # best solution found
    fbest = fcur
    T = T0                                       # temperature
    while T > Tmin:
        for l in range(L):                       # L moves at temperature T
            k = int(rand(r) * M)                 # random neighbour
            delta = two_opt_delta(cur, d, mi[k], mj[k])
            v = rand(r)
            if delta <= 0 or v < math.exp(-delta / T):   # Metropolis acceptance rule
                apply_two_opt(cur, mi[k], mj[k])
                fcur = fcur + delta
                if fcur < fbest:
                    best, fbest = cur[:], fcur
            if curve is not None:
                curve.append(fcur)
        T = alpha * T                            # cooling
    pi[:] = best
    return fbest

In [7]:
def simulated_annealing_kp(p, w, c, x, beta, T0, alpha, L, Tmin, r):
    """Simulated annealing for the KP with the flip neighbourhood and the penalized objective
    f(x) = profit(x) - beta * max{0, weight(x) - c}; infeasible solutions may be visited,
    the best solution is updated with feasible solutions only.
    x: feasible starting solution on input, best solution found on output (modified in place);
    returns its profit."""
    n = len(p)
    excess = weight(w, x) - c
    f = profit(p, x) - beta * (excess if excess > 0 else 0)
    best = x[:]
    zbest = profit(p, x)
    T = T0
    while T > Tmin:
        for l in range(L):
            k = int(rand(r) * n)                 # random flip
            y = x[:]
            y[k] = 1 - y[k]
            ey = weight(w, y) - c
            fy = profit(p, y) - beta * (ey if ey > 0 else 0)
            # maximization: delta > 0 is a worsening
            delta = f - fy
            v = rand(r)
            if delta <= 0 or v < math.exp(-delta / T):
                x[:] = y
                f = fy
                if ey <= 0 and profit(p, x) > zbest:
                    best, zbest = x[:], profit(p, x)
        T = alpha * T
    x[:] = best
    return zbest

## Tabu search

Tabu search for $P\,||\,C_{\max}$ and for the TSP.

In [8]:
def tabu_search_pcmax(t, m, mach, tenure, maxIter):
    """Tabu search for P||Cmax with the move neighbourhood (one job to another machine);
    the best non-tabu move is performed even if it is worse (makespan first, then sum of squares);
    after moving job j away from machine h, the attribute (j,h) is tabu for tenure iterations;
    a tabu move is allowed if it gives a makespan smaller than the best one (aspiration).
    mach: starting assignment on input, best assignment on output (modified in place);
    returns its makespan."""
    n = len(t)
    L = loads(mach, t, m)
    # tabu[j][h] >= it: j may not go to h
    tabu = [[0] * m for j in range(n)]
    best = mach[:]
    bestC, total, LB = 0, 0, 0
    for h in range(m):
        if L[h] > bestC:
            bestC = L[h]
    for j in range(n):
        total = total + t[j]
        if t[j] > LB:
            LB = t[j]
    # LB = max{ max t_j, ceil(sum/m) }
    if (total + m - 1) // m > LB:
        LB = (total + m - 1) // m
    it = 0
    while it < maxIter and bestC > LB:
        it = it + 1
        bj, bh = -1, -1
        bL = []
        for j in range(n):
            for h in range(m):
                if h == mach[j]:
                    continue
                L2 = L[:]
                L2[mach[j]] = L2[mach[j]] - t[j]
                L2[h] = L2[h] + t[j]
                C2 = 0
                for k in range(m):
                    if L2[k] > C2:
                        C2 = L2[k]
                # tabu and no aspiration
                if tabu[j][h] >= it and not (C2 < bestC):
                    continue
                if bj < 0 or lex_less(L2, bL):
                    bj, bh, bL = j, h, L2
        if bj < 0:
            break
        # j may not go back for tenure iterations
        tabu[bj][mach[bj]] = it + tenure
        mach[bj] = bh
        L = bL
        C = 0
        for k in range(m):
            if L[k] > C:
                C = L[k]
        if C < bestC:
            bestC, best = C, mach[:]
    mach[:] = best
    return bestC

In [9]:
def tabu_search_tsp(d, pi, tenure, maxIter):
    """Tabu search for the TSP with the 2-opt neighbourhood: the two edges removed by a move may
    not be added back for tenure iterations, unless the move gives a tour shorter than the best
    one (aspiration criterion). pi modified in place (best tour); returns its length."""
    n = len(pi)
    cur = pi[:]
    lcur = tour_length(cur, d)
    best = cur[:]
    lbest = lcur
    # tabu[a][b] >= it: edge (a,b) is tabu
    tabu = [[0] * n for a in range(n)]
    for it in range(1, maxIter + 1):
        bdelta, bi, bj = 0, -1, -1
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:
                    continue
                a, b, c, e = cur[i], cur[i + 1], cur[j], cur[(j + 1) % n]
                delta = d[a][c] + d[b][e] - d[a][b] - d[c][e]
                is_tabu = tabu[a][c] >= it or tabu[b][e] >= it
                if is_tabu and not (lcur + delta < lbest):
                    continue
                if bi < 0 or delta < bdelta:
                    bdelta, bi, bj = delta, i, j
        if bi < 0:
            break                                # every move is tabu
        a, b, c, e = cur[bi], cur[bi + 1], cur[bj], cur[(bj + 1) % n]
        tabu[a][b] = tabu[b][a] = it + tenure    # removed edges become tabu
        tabu[c][e] = tabu[e][c] = it + tenure
        # best non-tabu move, even if worse
        apply_two_opt(cur, bi, bj)
        lcur = lcur + bdelta
        if lcur < lbest:
            best, lbest = cur[:], lcur
    pi[:] = best
    return lbest

## Iterated local search and variable neighbourhood search

Iterated local search for the TSP with the double-bridge kick, and variable neighbourhood search for $P\,||\,C_{\max}$.

In [10]:
def double_bridge(pi, r):
    """Double-bridge kick: the tour A B C D, cut at the positions p1 < p2 < p3, becomes A C B D;
    returns the new tour."""
    n = len(pi)
    p1 = 1 + int(rand(r) * (n - 3))
    p2 = p1 + 1 + int(rand(r) * (n - 2 - p1))
    p3 = p2 + 1 + int(rand(r) * (n - 1 - p2))
    kick = []
    for k in range(0, p1):
        kick.append(pi[k])                       # A
    for k in range(p2, p3):
        kick.append(pi[k])                       # C
    for k in range(p1, p2):
        kick.append(pi[k])                       # B
    for k in range(p3, n):
        kick.append(pi[k])                       # D
    return kick


def iterated_local_search_tsp(d, pi, maxIter, r):
    """Iterated local search for the TSP: double-bridge kick + 2-opt local search, the new local
    optimum is accepted if it is better than the current one. pi modified in place; returns its length."""
    cur = pi[:]
    two_opt_ls(cur, d)                           # first local optimum
    lcur = tour_length(cur, d)
    for it in range(1, maxIter + 1):
        nw = double_bridge(cur, r)               # perturbation
        two_opt_ls(nw, d)                        # local search
        lnew = tour_length(nw, d)
        if lnew < lcur:                          # acceptance criterion
            cur, lcur = nw, lnew
    pi[:] = cur
    return lcur

In [11]:
def shake(t, m, mach, k, r):
    """Shaking in the k-th neighbourhood: k random moves, each one moving a random job j to a
    random machine different from mach[j]; returns the new assignment."""
    n = len(t)
    sh = mach[:]
    for s in range(k):
        j = int(rand(r) * n)
        g = int(rand(r) * (m - 1))
        # the (g+1)-th machine other than sh[j]
        sh[j] = g if g < sh[j] else g + 1
    return sh


def vns_pcmax(t, m, mach, kmax, maxIter, r):
    """Basic variable neighbourhood search for P||Cmax: shaking with k random moves, local search
    by move_descent; k returns to 1 after an improvement, otherwise it grows up to kmax.
    mach modified in place; returns its makespan."""
    n, total, LB = len(t), 0, 0
    for j in range(n):
        total = total + t[j]
        if t[j] > LB:
            LB = t[j]
    if (total + m - 1) // m > LB:
        LB = (total + m - 1) // m
    move_descent(t, m, mach)
    L = loads(mach, t, m)
    C = 0
    for h in range(m):
        if L[h] > C:
            C = L[h]
    it, k = 0, 1
    while it < maxIter and C > LB:
        it = it + 1
        nw = shake(t, m, mach, k, r)             # shaking
        move_descent(t, m, nw)                   # local search
        Lnew = loads(nw, t, m)
        if lex_less(Lnew, L):                    # move or not
            mach[:] = nw
            L, k = Lnew, 1
            C = 0
            for h in range(m):
                if L[h] > C:
                    C = L[h]
        else:
            k = k + 1 if k < kmax else 1
    return C

## GRASP

GRASP for the KP and for the TSP.

In [12]:
def randomized_greedy_kp(p, w, c, alpha, r):
    """Greedy randomized construction for the KP: at each step the candidates are the items that
    fit, RCL = { i candidate : p_i/w_i >= emax - alpha (emax - emin) }, sorted by non-increasing
    efficiency (ties: smallest index), and the item is chosen at random in the RCL; returns x."""
    n, rc = len(p), c                            # rc: residual capacity
    x = [0] * n
    while True:
        emax, emin = -1, -1
        for i in range(n):
            if x[i] == 1 or w[i] > rc:
                continue
            e = p[i] / w[i]
            if emax < 0 or e > emax:
                emax = e
            if emin < 0 or e < emin:
                emin = e
        if emax < 0:
            return x                             # no item fits
        thr = emax - alpha * (emax - emin)
        rcl = []
        for i in range(n):
            if x[i] == 1 or w[i] > rc or p[i] / w[i] < thr - 1e-9:
                continue
            pos = len(rcl)                       # insertion in sorted position
            rcl.append(i)
            while pos > 0 and p[rcl[pos - 1]] * w[i] < p[i] * w[rcl[pos - 1]]:
                rcl[pos] = rcl[pos - 1]
                pos = pos - 1
            rcl[pos] = i
        i = rcl[int(rand(r) * len(rcl))]
        x[i] = 1
        rc = rc - w[i]


def grasp_kp(p, w, c, alpha, maxIter, r):
    """GRASP for the KP: greedy randomized construction + local search, maxIter times.
    Returns (zbest, best): the best profit and the best solution."""
    zbest = -1
    best = []
    for it in range(1, maxIter + 1):
        x = randomized_greedy_kp(p, w, c, alpha, r)  # construction
        kp_local_search(p, w, c, x)              # local search
        z = profit(p, x)
        if z > zbest:
            best, zbest = x, z
    return zbest, best

In [13]:
def randomized_nn(d, alpha, r):
    """Randomized nearest neighbour from city 0: the next city is chosen at random in the
    restricted candidate list RCL = { j unvisited : d[last][j] <= dmin + alpha (dmax - dmin) },
    sorted by non-decreasing distance (ties: smallest index); returns the tour."""
    n = len(d)
    pi = [0]
    visited = [False] * n
    visited[0] = True
    for k in range(1, n):
        last = pi[-1]
        dmin, dmax = -1, -1
        for j in range(n):
            if visited[j]:
                continue
            if dmin < 0 or d[last][j] < dmin:
                dmin = d[last][j]
            if dmax < 0 or d[last][j] > dmax:
                dmax = d[last][j]
        thr = dmin + alpha * (dmax - dmin)
        rcl = []
        for j in range(n):
            if visited[j] or d[last][j] > thr + 1e-9:
                continue
            pos = len(rcl)                       # insertion in sorted position
            rcl.append(j)
            while pos > 0 and d[last][rcl[pos - 1]] > d[last][j]:
                rcl[pos] = rcl[pos - 1]
                pos = pos - 1
            rcl[pos] = j
        nxt = rcl[int(rand(r) * len(rcl))]
        pi.append(nxt)
        visited[nxt] = True
    return pi


def grasp_tsp(d, alpha, maxIter, r):
    """GRASP for the TSP: randomized nearest neighbour + 2-opt local search, maxIter times.
    Returns (lbest, best): the best length and the best tour."""
    lbest = -1
    best = []
    for it in range(1, maxIter + 1):
        pi = randomized_nn(d, alpha, r)          # greedy randomized construction
        two_opt_ls(pi, d)                        # local search
        length = tour_length(pi, d)
        if lbest < 0 or length < lbest:
            best, lbest = pi, length
    return lbest, best

## Large neighbourhood search

LNS and ALNS for the TSP; LNS for the BPP, with the First Fit starting packing, the destroy method and the two repair methods.

In [14]:
def random_removal(pi, q, r):
    """Destroy: removes q random cities from the tour pi (never the first city; pi modified in
    place) and returns them."""
    removed = []
    for s in range(q):
        k = 1 + int(rand(r) * (len(pi) - 1))
        removed.append(pi[k])
        del pi[k]
    return removed


def worst_removal(pi, q, d):
    """Destroy: removes, q times, the city whose removal saves the most length (never the first
    city; pi modified in place) and returns the removed cities."""
    removed = []
    for s in range(q):
        n, bk, bs = len(pi), -1, 0
        for k in range(1, n):
            a, v, b = pi[k - 1], pi[k], pi[(k + 1) % n]
            saving = d[a][v] + d[v][b] - d[a][b]
            if bk < 0 or saving > bs:
                bk, bs = k, saving
        removed.append(pi[bk])
        del pi[bk]
    return removed


def cheapest_insertion_repair(pi, removed, d):
    """Repair: inserts the removed cities, in the order of removal, each in the position of
    minimum insertion cost d[a][v] + d[v][b] - d[a][b] (ties: first position); pi modified in place."""
    for s in range(len(removed)):
        v, n, bpos, bcost = removed[s], len(pi), -1, 0
        for k in range(n):
            a, b = pi[k], pi[(k + 1) % n]
            cost = d[a][v] + d[v][b] - d[a][b]
            if bpos < 0 or cost < bcost:
                bpos, bcost = k, cost
        pi.insert(bpos + 1, v)

In [15]:
def lns_tsp(d, pi, q, maxIter, r):
    """Large neighbourhood search for the TSP: random removal of q cities + cheapest insertion,
    the new tour is accepted if it is shorter than the current one.
    pi modified in place; returns its length."""
    cur = pi[:]
    lcur = tour_length(cur, d)
    for it in range(1, maxIter + 1):
        nw = cur[:]
        removed = random_removal(nw, q, r)       # destroy
        cheapest_insertion_repair(nw, removed, d)  # repair
        lnew = tour_length(nw, d)
        if lnew < lcur:                          # acceptance
            cur, lcur = nw, lnew
    pi[:] = cur
    return lcur


def alns_tsp(d, pi, q, maxIter, rho, r, wgt=None):
    """Adaptive large neighbourhood search for the TSP with two destroy operators
    (o = 0: random removal, o = 1: worst removal) and cheapest insertion as repair;
    the operator is chosen with probability proportional to its weight wgt[o], updated as
    wgt[o] = (1 - rho) wgt[o] + rho psi, with psi = 3, 2, 1, 0 if the new tour is a new best,
    better than the current one, as good as the current one, worse.
    pi modified in place (best tour); returns its length; if the list wgt is given, it receives
    the final weights."""
    if wgt is None:
        wgt = [0.0, 0.0]
    cur = pi[:]
    lcur = tour_length(cur, d)
    best = cur[:]
    lbest = lcur
    wgt[0] = 1.0
    wgt[1] = 1.0
    for it in range(1, maxIter + 1):
        o = 0 if rand(r) * (wgt[0] + wgt[1]) < wgt[0] else 1   # roulette wheel
        nw = cur[:]
        removed = random_removal(nw, q, r) if o == 0 else worst_removal(nw, q, d)
        cheapest_insertion_repair(nw, removed, d)
        lnew = tour_length(nw, d)
        if lnew < lbest:
            psi = 3.0
        elif lnew < lcur:
            psi = 2.0
        elif lnew == lcur:
            psi = 1.0
        else:
            psi = 0.0
        if lnew <= lcur:
            cur, lcur = nw, lnew
        if lcur < lbest:
            best, lbest = cur[:], lcur
        wgt[o] = (1 - rho) * wgt[o] + rho * psi                 # adaptive weights
    pi[:] = best
    return lbest

In [16]:
def first_fit(w, c, order):
    """First fit on the items in the given order.
    Returns (nb, bin): the number of bins and bin[i] = bin of item i."""
    load = []
    bin = [-1] * len(w)
    for a in range(len(order)):
        i, b = order[a], 0
        while b < len(load) and load[b] + w[i] > c:
            b = b + 1
        if b == len(load):
            load.append(0)                       # open a new bin
        bin[i] = b
        load[b] = load[b] + w[i]
    return len(load), bin


def destroy_bins(bin, nb, w, q, r):
    """Destroy for the BPP: the least loaded bin (ties: smallest index) and q-1 other bins at
    random; returns the list of the chosen bins."""
    load = [0] * nb
    for i in range(len(w)):
        load[bin[i]] = load[bin[i]] + w[i]
    e = 0
    for b in range(1, nb):
        if load[b] < load[e]:
            e = b
    empty = [e]
    while len(empty) < q and len(empty) < nb:
        others = []
        for b in range(nb):
            chosen = False
            for s in range(len(empty)):
                if empty[s] == b:
                    chosen = True
            if not chosen:
                others.append(b)
        empty.append(others[int(rand(r) * len(others))])
    return empty

In [17]:
def repair_ffd(items, w, c):
    """Heuristic repair for the BPP: first fit decreasing on the removed items, in new bins.
    Returns (k, asg): the number of new bins and asg[s] = bin (0, 1, ...) of items[s]."""
    k = len(items)
    order = [0] * k
    for s in range(k):
        order[s] = s
    for a in range(1, k):                        # non-increasing weight, stable
        s, b = order[a], a
        while b > 0 and w[items[order[b - 1]]] < w[items[s]]:
            order[b] = order[b - 1]
            b = b - 1
        order[b] = s
    load = []
    asg = [-1] * k
    for a in range(k):
        s, b = order[a], 0
        while b < len(load) and load[b] + w[items[s]] > c:
            b = b + 1
        if b == len(load):
            load.append(0)
        asg[s] = b
        load[b] = load[b] + w[items[s]]
    return len(load), asg


def repair_exact(items, w, c):
    """Exact repair for the BPP: optimal packing of the removed items in new bins, by a
    depth-first branch and bound (items by non-increasing weight; each item goes in an open bin
    or in a new one). Returns (k, asg) as repair_ffd."""
    k = len(items)
    order = [0] * k
    for s in range(k):
        order[s] = s
    for a in range(1, k):
        s, b = order[a], a
        while b > 0 and w[items[order[b - 1]]] < w[items[s]]:
            order[b] = order[b - 1]
            b = b - 1
        order[b] = s
    load, cur = [], [-1] * k
    best = [k + 1]                               # best number of bins (a list: changed by branch)
    asg = [-1] * k

    def branch(a):
        if len(load) >= best[0]:
            return                               # bound: cannot improve
        if a == len(order):
            best[0] = len(load)
            asg[:] = cur
            return
        s = order[a]
        for b in range(len(load)):
            if load[b] + w[items[s]] > c:
                continue
            load[b] = load[b] + w[items[s]]
            cur[s] = b
            branch(a + 1)
            load[b] = load[b] - w[items[s]]
        load.append(w[items[s]])
        cur[s] = len(load) - 1
        branch(a + 1)
        load.pop()

    branch(0)
    return best[0], asg


def lns_bpp(w, c, bin, nb, q, exact, maxIter, r):
    """Large neighbourhood search for the BPP: empty q bins (destroy_bins) and repack their items
    with FFD (exact = False) or optimally (exact = True); the new packing is accepted if it uses
    fewer bins, or the same number of bins and a sum of squared loads not smaller.
    bin, nb: starting packing; bin is modified in place; returns the final number of bins."""
    n, total = len(w), 0
    for i in range(n):
        total = total + w[i]
    LB = (total + c - 1) // c
    it = 0
    while it < maxIter and nb > LB:
        it = it + 1
        empty = destroy_bins(bin, nb, w, q, r)   # destroy
        is_emptied = [False] * nb
        for s in range(len(empty)):
            is_emptied[empty[s]] = True
        items, newbin, new_index = [], [-1] * n, [-1] * nb
        kept = 0
        for b in range(nb):
            if not is_emptied[b]:
                new_index[b] = kept
                kept = kept + 1
        for i in range(n):
            if is_emptied[bin[i]]:
                items.append(i)
            else:
                newbin[i] = new_index[bin[i]]
        # repair
        k, asg = repair_exact(items, w, c) if exact else repair_ffd(items, w, c)
        for s in range(len(items)):
            newbin[items[s]] = kept + asg[s]
        nb2 = kept + k
        load, load2 = [0] * nb, [0] * nb2
        for i in range(n):
            load[bin[i]] += w[i]
            load2[newbin[i]] += w[i]
        # auxiliary objective: sum of squared loads
        aux, aux2 = 0, 0
        for b in range(nb):
            aux = aux + load[b] * load[b]
        for b in range(nb2):
            aux2 = aux2 + load2[b] * load2[b]
        # acceptance
        if nb2 < nb or (nb2 == nb and aux2 >= aux):
            bin[:] = newbin
            nb = nb2
    return nb

## Genetic and memetic algorithm

The genetic and memetic algorithm for the KP.

In [18]:
def repair_kp(p, w, c, x):
    """Repair of an infeasible KP solution: removes the selected item of minimum efficiency
    (ties: largest index) until the capacity is respected; x modified in place."""
    n, W = len(p), weight(w, x)
    while W > c:
        s = -1
        for i in range(n):
            if x[i] == 1 and (s < 0 or p[i] * w[s] <= p[s] * w[i]):
                s = i
        x[s] = 0
        W = W - w[s]


def tournament(fit, r):
    """Binary tournament: two random individuals, the fitter one wins (ties: the first drawn)."""
    P = len(fit)
    a = int(rand(r) * P)
    b = int(rand(r) * P)
    return a if fit[a] >= fit[b] else b


def genetic_kp(p, w, c, P, pm, maxIter, memetic, r):
    """Steady-state genetic algorithm for the KP with binary encoding: binary tournament
    selection, one-point crossover, bit-flip mutation with probability pm, repair; if memetic is
    True, every new individual is also improved by kp_local_search; the child replaces the worst
    individual (ties: last index) if it is better and not already in the population.
    Returns (zbest, best): the best profit and the best individual."""
    n = len(p)
    pop = [[0] * n for s in range(P)]
    fit = [0] * P
    for s in range(P):                           # random initial population
        for i in range(n):
            pop[s][i] = 1 if rand(r) < 0.5 else 0
        repair_kp(p, w, c, pop[s])
        if memetic:
            kp_local_search(p, w, c, pop[s])
        fit[s] = profit(p, pop[s])
    for it in range(1, maxIter + 1):
        a = tournament(fit, r)                   # selection
        b = tournament(fit, r)
        cut = 1 + int(rand(r) * (n - 1))         # one-point crossover
        child = [0] * n
        for i in range(n):
            child[i] = pop[a][i] if i < cut else pop[b][i]
        for i in range(n):                       # mutation
            if rand(r) < pm:
                child[i] = 1 - child[i]
        repair_kp(p, w, c, child)                # repair
        if memetic:
            kp_local_search(p, w, c, child)      # memetic: local search
        fc = profit(p, child)
        worst = 0
        for s in range(1, P):
            if fit[s] <= fit[worst]:
                worst = s
        present = False
        for s in range(P):
            if pop[s] == child:
                present = True
        # replacement
        if not present and fc > fit[worst]:
            pop[worst] = child
            fit[worst] = fc
    sb = 0
    for s in range(1, P):
        if fit[s] > fit[sb]:
            sb = s
    return fit[sb], pop[sb][:]

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [19]:
# runs every example of the lecture notes and every instance of the exercises, and compares the
# value found with the one computed by an independent implementation

errors = 0
checks = 0


def check(name, found, expected):
    global errors, checks
    checks += 1
    if found != expected:
        errors += 1
    print("%-46s %6d %s" % (name, found, "OK" if found == expected else "<-- EXPECTED A DIFFERENT VALUE"))


def S(seed):
    return Random(seed)


def G(g):
    return Random(1, g)


def makespan(L):
    C = 0
    for x in L:
        if x > C:
            C = x
    return C


dR = [[0, 5, 2, 2, 9], [5, 0, 4, 3, 4], [2, 4, 0, 4, 7], [2, 3, 4, 0, 7], [9, 4, 7, 7, 0]]
d20 = [[0, 58, 33, 55, 79, 83, 30, 53, 72, 86, 38, 88, 46, 92, 23, 21, 63, 36, 35, 93],
       [58, 0, 33, 14, 28, 37, 29, 18, 35, 28, 42, 37, 21, 36, 53, 45, 10, 27, 23, 36],
       [33, 33, 0, 39, 48, 51, 9, 22, 40, 59, 44, 56, 15, 63, 20, 34, 34, 6, 22, 65],
       [55, 14, 39, 0, 42, 51, 32, 31, 49, 37, 30, 50, 31, 46, 57, 38, 24, 33, 21, 45],
       [79, 28, 48, 42, 0, 10, 49, 26, 18, 20, 70, 9, 33, 17, 67, 71, 19, 43, 49, 22],
       [83, 37, 51, 51, 10, 0, 54, 31, 14, 29, 78, 9, 37, 22, 69, 77, 27, 47, 56, 29],
       [30, 29, 9, 32, 49, 54, 0, 23, 44, 57, 35, 58, 16, 62, 25, 26, 33, 7, 13, 64],
       [53, 18, 22, 31, 26, 31, 23, 0, 22, 38, 50, 35, 7, 41, 42, 47, 14, 17, 26, 44],
       [72, 35, 40, 49, 18, 14, 44, 22, 0, 38, 72, 22, 28, 34, 56, 69, 26, 37, 49, 40],
       [86, 28, 59, 37, 20, 29, 57, 38, 38, 0, 67, 23, 44, 11, 79, 73, 24, 53, 51, 8],
       [38, 42, 44, 30, 70, 78, 35, 50, 72, 67, 0, 79, 46, 77, 52, 17, 51, 41, 24, 75],
       [88, 37, 56, 50, 9, 9, 58, 35, 22, 23, 79, 0, 42, 14, 75, 80, 28, 52, 58, 21],
       [46, 21, 15, 31, 33, 37, 16, 7, 28, 44, 46, 42, 0, 48, 35, 42, 20, 10, 22, 51],
       [92, 36, 63, 46, 17, 22, 62, 41, 34, 11, 77, 14, 48, 0, 82, 81, 29, 57, 59, 7],
       [23, 53, 20, 57, 67, 69, 25, 42, 56, 79, 52, 75, 35, 82, 0, 37, 55, 26, 37, 85],
       [21, 45, 34, 38, 71, 77, 26, 47, 69, 73, 17, 80, 42, 81, 37, 0, 53, 33, 22, 81],
       [63, 10, 34, 24, 19, 27, 33, 14, 26, 24, 51, 28, 20, 29, 55, 53, 0, 29, 30, 31],
       [36, 27, 6, 33, 43, 47, 7, 17, 37, 53, 41, 52, 10, 57, 26, 33, 29, 0, 17, 60],
       [35, 23, 22, 21, 49, 56, 13, 26, 49, 51, 24, 58, 22, 59, 37, 22, 30, 17, 0, 59],
       [93, 36, 65, 45, 22, 29, 64, 44, 40, 8, 75, 21, 51, 7, 85, 81, 31, 60, 59, 0]]
pR, wR, cR = [10, 7, 6, 4], [5, 4, 3, 3], 9
p20 = [54, 55, 55, 52, 58, 42, 53, 53, 22, 64, 54, 51, 39, 49, 53, 46, 28, 64, 58, 34]
w20 = [39, 43, 49, 47, 44, 35, 43, 44, 16, 50, 40, 42, 28, 38, 46, 39, 14, 50, 45, 27]
c20 = 389
tR, mR = [5, 5, 4, 4, 3, 3, 3], 3
t15, m15 = [10, 23, 26, 16, 23, 11, 14, 18, 18, 24, 29, 22, 13, 13, 15], 4
wB, cB = [4, 4, 5, 3, 2, 3], 7
w24 = [30, 40, 57, 22, 37, 23, 34, 34, 55, 51, 21, 47, 22, 24, 22, 34, 20, 51, 25, 54, 35, 27, 55, 48]
c24 = 100
nnR, nn20 = nearest_neighbour(dR), nearest_neighbour(d20)
ord24 = [2, 8, 22, 19, 9, 17, 23, 11, 1, 4, 20, 6, 7, 15, 0, 21, 18, 13, 5, 3, 12, 14, 10, 16]
seeds = range(1, 11)

# examples of the lecture notes on the small instances
check("TSP ref: nearest neighbour", tour_length(nnR, dR), 25)
check("Ex SA TSP ref (seed 1)", simulated_annealing_tsp(dR, nnR[:], 5.0, 0.8, 2, 3.0, S(1)), 18)
check("Ex SA KP ref (seed 3)", simulated_annealing_kp(pR, wR, cR, [1, 0, 1, 0], 3, 4.0, 0.8, 2, 2.5, S(3)), 17)
for tenure, expected in zip(range(7), [11, 11, 11, 11, 10, 10, 9]):
    check("Ex TS PCmax ref tenure=%d" % tenure, tabu_search_pcmax(tR, mR, lpt(tR, mR), tenure, 50), expected)
check("Ex VNS PCmax ref (seed 1)", vns_pcmax(tR, mR, lpt(tR, mR), 3, 50, S(1)), 9)
check("Ex GRASP KP ref (seed 2)", grasp_kp(pR, wR, cR, 0.5, 2, S(2))[0], 17)
check("Ex LNS TSP ref (seed 1)", lns_tsp(dR, nnR[:], 2, 1, S(1)), 18)
nb, bin = first_fit(wB, cB, [0, 1, 2, 3, 4, 5])
check("Ex LNS BPP ref (seed 3)", lns_bpp(wB, cB, bin, nb, 2, False, 10, S(3)), 3)
check("Ex GA KP ref (seed 36)", genetic_kp(pR, wR, cR, 4, 0.25, 3, False, S(36))[0], 17)

# TSP with 20 cities
check("TSP20 nearest neighbour", tour_length(nn20, d20), 326)
pi = nn20[:]
two_opt_ls(pi, d20)
check("TSP20 2-opt", tour_length(pi, d20), 319)
sa_expected = {
    (0.9, 1): [299, 319, 305, 291, 305, 319, 322, 319, 319, 305],
    (0.9, 5): [291, 291, 291, 298, 298, 305, 305, 291, 298, 305],
    (0.9, 20): [294, 304, 298, 310, 323, 291, 291, 310, 291, 291],
    (0.9, 100): [291, 291, 298, 298, 291, 311, 303, 298, 299, 298],
    (0.95, 1): [299, 291, 305, 291, 305, 319, 291, 298, 319, 305],
    (0.95, 5): [291, 291, 298, 298, 298, 291, 291, 291, 298, 296],
    (0.95, 20): [298, 298, 296, 298, 291, 291, 291, 298, 291, 291],
    (0.95, 100): [298, 291, 299, 291, 291, 291, 291, 291, 291, 298],
    (0.99, 1): [299, 305, 305, 291, 305, 305, 291, 291, 291, 305],
    (0.99, 5): [291, 291, 291, 291, 298, 291, 291, 291, 291, 291],
    (0.99, 20): [291, 291, 291, 291, 291, 291, 291, 291, 291, 291],
    (0.99, 100): [291, 291, 291, 291, 291, 291, 291, 291, 291, 291],
}
for alpha in [0.9, 0.95, 0.99]:
    for T0 in [1.0, 5.0, 20.0, 100.0]:
        for seed in seeds:
            check("TSP20 SA alpha=%g T0=%g seed %d" % (alpha, T0, seed),
                  simulated_annealing_tsp(d20, nn20[:], T0, alpha, 100, 0.5, S(seed)),
                  sa_expected[(alpha, T0)][seed - 1])
for tenure in [0, 1, 2, 5, 10, 20]:
    check("TSP20 TS tenure=%d" % tenure, tabu_search_tsp(d20, nn20[:], tenure, 200), 319 if tenure == 0 else 291)
for seed in seeds:
    check("TSP20 ILS seed %d" % seed, iterated_local_search_tsp(d20, nn20[:], 100, S(seed)), 291)

# GRASP on the TSP with 20 cities and on the KP with 20 items
grasp_expected_kp = {0.0: [514] * 10, 0.1: [521] * 10, 0.25: [521] * 10, 0.5: [521] * 10,
                     1.0: [521, 519, 521, 521, 521, 521, 521, 521, 521, 521]}
for alpha in [0.0, 0.1, 0.25, 0.5, 1.0]:
    for seed in seeds:
        check("TSP20 GRASP alpha=%g seed %d" % (alpha, seed), grasp_tsp(d20, alpha, 20, S(seed))[0],
              319 if alpha == 0.0 else 291)
        check("KP20 GRASP alpha=%g seed %d" % (alpha, seed), grasp_kp(p20, w20, c20, alpha, 20, S(seed))[0],
              grasp_expected_kp[alpha][seed - 1])

# LNS and ALNS on the TSP with 20 cities
lns_expected = {2: [319] * 10, 3: [319] * 10, 5: [291, 291, 291, 319, 291, 291, 291, 291, 291, 291],
                8: [291] * 10}
for q in [2, 3, 5, 8]:
    for seed in seeds:
        check("TSP20 LNS q=%d seed %d" % (q, seed), lns_tsp(d20, nn20[:], q, 200, S(seed)), lns_expected[q][seed - 1])
alns_expected = [291, 291, 291, 291, 319, 319, 291, 291, 291, 319]
alns_expected_w = [846, 971, 768, 803, 839, 980, 946, 827, 912, 793]
for seed in seeds:
    check("TSP20 ALNS seed %d" % seed, alns_tsp(d20, nn20[:], 5, 200, 0.1, S(seed)), alns_expected[seed - 1])
    wg = [0.0, 0.0]
    alns_tsp(d20, nn20[:], 5, 200, 0.1, S(seed), wg)
    check("TSP20 ALNS seed %d weight1 x1000" % seed, int(1000 * wg[0]), alns_expected_w[seed - 1])

# genetic and memetic algorithm on the KP with 20 items
ga_expected = [521, 519, 521, 508, 514, 521, 510, 519, 521, 515]
for seed in seeds:
    check("KP20 GA seed %d" % seed, genetic_kp(p20, w20, c20, 10, 0.05, 300, False, S(seed))[0], ga_expected[seed - 1])
for seed in seeds:
    check("KP20 memetic seed %d" % seed, genetic_kp(p20, w20, c20, 10, 0.05, 300, True, S(seed))[0], 521)

# P||Cmax with 15 jobs and 4 machines
check("PCM15 LPT", makespan(loads(lpt(t15, m15), t15, m15)), 73)
for tenure, expected in zip([0, 1, 2, 3, 5, 7, 10, 15, 20, 30], [73, 73, 73, 73, 73, 72, 71, 69, 70, 70]):
    check("PCM15 TS tenure=%d" % tenure, tabu_search_pcmax(t15, m15, lpt(t15, m15), tenure, 300), expected)
vns_expected = {1: [69, 70, 70, 70, 69, 69, 69, 69, 70, 69],
                2: [70, 69, 69, 69, 69, 70, 69, 70, 70, 69],
                3: [70, 69, 69, 69, 69, 69, 69, 70, 70, 69]}
for kmax in [1, 2, 3]:
    for seed in seeds:
        check("PCM15 VNS kmax=%d seed %d" % (kmax, seed), vns_pcmax(t15, m15, lpt(t15, m15), kmax, 300, S(seed)),
              vns_expected[kmax][seed - 1])

# BPP with 24 items
check("BPP24 FFD", first_fit(w24, c24, ord24)[0], 10)
for q in [2, 3, 4]:
    for exact in [False, True]:
        for seed in seeds:
            nb, bin = first_fit(w24, c24, ord24)
            check("BPP24 LNS q=%d %s seed %d" % (q, "exact" if exact else "FFD", seed),
                  lns_bpp(w24, c24, bin, nb, q, exact, 200, S(seed)), 9 if (exact and q >= 3) else 10)

# exercises
d6 = [[0, 5, 1, 7, 7, 4], [5, 0, 4, 5, 2, 1], [1, 4, 0, 6, 6, 4], [7, 5, 6, 0, 7, 6], [7, 2, 6, 7, 0, 2],
      [4, 1, 4, 6, 2, 0]]
pi6 = [0, 2, 1, 5, 4, 3]
p5, w5, c5 = [11, 5, 18, 8, 8], [5, 3, 11, 6, 7], 16
g_sa = [0.13, 0.76, 0.46, 0.53, 0.22, 0.05, 0.68, 0.68, 0.93, 0.38, 0.52, 0.83]
check("EX_SA_1", simulated_annealing_tsp(d6, pi6[:], 5.0, 0.8, 2, 3.0, G(g_sa)), 20)
g_sa2 = [0.63, 0.11, 0.17, 0.66, 0.38, 0.94, 0.58, 0.59, 0.69, 0.67, 0.39, 0.62, 0.69, 0.07, 0.59, 0.42]
check("EX_SA_2 beta=1", simulated_annealing_kp(p5, w5, c5, [1, 1, 0, 1, 0], 1, 4.0, 0.8, 2, 1.7, G(g_sa2)), 24)
check("EX_SA_2 beta=3", simulated_annealing_kp(p5, w5, c5, [1, 1, 0, 1, 0], 3, 4.0, 0.8, 2, 1.7, G(g_sa2)), 29)
check("EX_SA_3", simulated_annealing_tsp(d6, pi6[:], 0.5, 0.8, 2, 0.3, G(g_sa)), 22)
t5 = [3, 3, 2, 2, 2]
check("EX_TS_1", tabu_search_pcmax(t5, 2, lpt(t5, 2), 3, 20), 6)
check("EX_TS_2 tenure=0", tabu_search_pcmax(t5, 2, lpt(t5, 2), 0, 50), 7)
check("EX_TS_2 tenure=1", tabu_search_pcmax(t5, 2, lpt(t5, 2), 1, 50), 7)
check("EX_TS_3", tabu_search_tsp(d6, pi6[:], 2, 3), 20)
check("EX_IV_1", iterated_local_search_tsp(d6, pi6[:], 1, G([0.13, 0.76, 0.46])), 20)
check("EX_IV_2", iterated_local_search_tsp(d6, pi6[:], 1, G([0.16, 0.13, 0.01])), 22)
t7 = [4, 4, 3, 3, 2, 2, 2]
check("EX_IV_3", vns_pcmax(t7, 3, lpt(t7, 3), 3, 10,
                           G([0.26, 0.51, 0.92, 0.07, 0.44, 0.09, 0.36, 0.36, 0.87, 0.77, 0.04, 0.66])), 7)
check("EX_GRASP_1", grasp_kp(p5, w5, c5, 0.5, 2, G([0.39, 0.27, 0.38, 0.6, 0.66, 0.14, 0.04, 0.04, 0.8, 0.15]))[0], 29)
check("EX_GRASP_2", grasp_tsp(d6, 0.5, 2, G([0.45, 0.31, 0.05, 0.86, 0.41, 0.52, 0.47, 0.47, 0.28, 0.22]))[0], 20)
g_gr3 = [0.92, 0.29, 0.21, 0.73, 0.53, 0.33, 0.75, 0.76, 0.54, 0.68, 0.64, 0.82]
check("EX_GRASP_3 alpha=0", grasp_kp(p5, w5, c5, 0.0, 2, G(g_gr3))[0], 24)
check("EX_GRASP_3 alpha=1", grasp_kp(p5, w5, c5, 1.0, 2, G(g_gr3))[0], 24)
w8 = [2, 7, 5, 5, 3, 2, 6, 6]
nb, bin = first_fit(w8, 10, [0, 1, 2, 3, 4, 5, 6, 7])
check("EX_LNS_1", lns_bpp(w8, 10, bin, nb, 2, False, 10,
                          G([0.13, 0.76, 0.46, 0.53, 0.22, 0.05, 0.68, 0.68, 0.93, 0.38])), 4)
w8 = [3, 3, 6, 5, 4, 4, 2, 3]
nb, bin = first_fit(w8, 10, [2, 3, 4, 5, 0, 1, 7, 6])
check("EX_LNS_2 exact", lns_bpp(w8, 10, bin, nb, 3, True, 1, G([0.66, 0.78, 0.29, 0.66])), 3)
nb, bin = first_fit(w8, 10, [2, 3, 4, 5, 0, 1, 7, 6])
check("EX_LNS_2 FFD", lns_bpp(w8, 10, bin, nb, 3, False, 1, G([0.66, 0.78, 0.29, 0.66])), 4)
check("EX_LNS_3", lns_tsp(d6, pi6[:], 2, 2, G([0.68, 0.16, 0.84, 0.92])), 20)
g_ga = [0.75, 0.75, 0.75, 0.25, 0.25, 0.75, 0.25, 0.75, 0.75, 0.25, 0.75, 0.25, 0.75, 0.25, 0.25, 0.25, 0.25,
        0.75, 0.25, 0.75, 0.58, 0.07, 0.5, 0.39, 0.63, 0.56, 0.15, 0.15, 0.22, 0.6, 0.23, 0.97, 0.41, 0.64,
        0.36, 0.05, 0.09, 0.6, 0.8, 0.01]
check("EX_GA_1", genetic_kp(p5, w5, c5, 4, 0.2, 2, False, G(g_ga))[0], 24)
check("EX_GA_2", genetic_kp(p5, w5, c5, 4, 0.2, 2, True, G(g_ga))[0], 29)
print("\n%d checks, %d errors" % (checks, errors))

TSP ref: nearest neighbour                         25 OK
Ex SA TSP ref (seed 1)                             18 OK
Ex SA KP ref (seed 3)                              17 OK
Ex TS PCmax ref tenure=0                           11 OK
Ex TS PCmax ref tenure=1                           11 OK
Ex TS PCmax ref tenure=2                           11 OK
Ex TS PCmax ref tenure=3                           11 OK
Ex TS PCmax ref tenure=4                           10 OK
Ex TS PCmax ref tenure=5                           10 OK
Ex TS PCmax ref tenure=6                            9 OK
Ex VNS PCmax ref (seed 1)                           9 OK
Ex GRASP KP ref (seed 2)                           17 OK
Ex LNS TSP ref (seed 1)                            18 OK
Ex LNS BPP ref (seed 3)                             3 OK
Ex GA KP ref (seed 36)                             17 OK
TSP20 nearest neighbour                           326 OK
TSP20 2-opt                                       319 OK
TSP20 SA alpha=0.9 T0=1 seed 1 

## Try it yourself

Change the data in the cell below and run it again.

In [20]:
# change the data below and run the cell again
d = [[0, 5, 1, 7, 7, 4], [5, 0, 4, 5, 2, 1], [1, 4, 0, 6, 6, 4],
     [7, 5, 6, 0, 7, 6], [7, 2, 6, 7, 0, 2], [4, 1, 4, 6, 2, 0]]   # TSP: symmetric distances
p, w, c = [11, 5, 18, 8, 8], [5, 3, 11, 6, 7], 16                  # KP: profits, weights, capacity
seed = 1                                                           # seed of the random numbers


def show(name, value, sol):
    print("%-22s %3d  [%s]" % (name, value, " ".join("%d" % v for v in sol)))


nn = nearest_neighbour(d)
show("TSP nearest neighbour", tour_length(nn, d), nn)
pi = nn[:]; show("TSP simulated annealing", simulated_annealing_tsp(d, pi, 10.0, 0.9, 20, 0.1, Random(seed)), pi)
pi = nn[:]; show("TSP tabu search", tabu_search_tsp(d, pi, 2, 50), pi)
pi = nn[:]; show("TSP iterated LS", iterated_local_search_tsp(d, pi, 20, Random(seed)), pi)
length, pi = grasp_tsp(d, 0.3, 10, Random(seed)); show("TSP GRASP", length, pi)
pi = nn[:]; show("TSP LNS", lns_tsp(d, pi, 2, 50, Random(seed)), pi)
x = [0] * len(p); show("KP simulated annealing", simulated_annealing_kp(p, w, c, x, 3, 10.0, 0.9, 20, 0.1, Random(seed)), x)
z, x = grasp_kp(p, w, c, 0.3, 10, Random(seed)); show("KP GRASP", z, x)
z, x = genetic_kp(p, w, c, 6, 0.1, 50, False, Random(seed)); show("KP genetic", z, x)
z, x = genetic_kp(p, w, c, 6, 0.1, 50, True, Random(seed)); show("KP memetic", z, x)

TSP nearest neighbour   22  [0 2 1 5 4 3]
TSP simulated annealing  20  [0 5 4 1 3 2]
TSP tabu search         20  [0 2 3 1 4 5]
TSP iterated LS         20  [0 2 3 1 4 5]
TSP GRASP               20  [0 2 3 1 4 5]
TSP LNS                 20  [0 5 4 1 3 2]
KP simulated annealing  29  [1 0 1 0 0]
KP GRASP                29  [1 0 1 0 0]
KP genetic              29  [1 0 1 0 0]
KP memetic              29  [1 0 1 0 0]


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).